In [ ]:
import os
import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.cluster import KMeans

np.random.seed(42)

print("Libraries loaded successfully.")

In [ ]:
import kagglehub

path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")
print("Path to dataset files:", path)

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

image_paths = [
    str(p) for p in Path(path).rglob("*")
    if p.is_file() and p.suffix.lower() in image_extensions
]

print(f"Number of image files found: {len(image_paths):,}")

if not image_paths:
    raise FileNotFoundError("No JPG/PNG image was found in the downloaded dataset.")

image_paths = sorted(image_paths)
image_path = image_paths[0]

print("Selected image:", image_path)

img_bgr = cv2.imread(image_path, cv2.IMREAD_COLOR)

if img_bgr is None:
    raise ValueError(f"OpenCV could not read the selected image: {image_path}")

print("Image loaded successfully.")

In [ ]:
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_gray_float = (
    0.299 * img_rgb[:, :, 0] +
    0.587 * img_rgb[:, :, 1] +
    0.114 * img_rgb[:, :, 2]
)

img_gray = np.clip(img_gray_float, 0, 255).astype(np.uint8)

h, w, c = img_bgr.shape

print("=" * 60)
print("IMAGE REPRESENTATION")
print("=" * 60)
print(f"Image path       : {image_path}")
print(f"Matrix dimensions: H={h}, W={w}, C={c}")
print(f"Data type        : {img_bgr.dtype}")
print(f"BGR intensity    : [{img_bgr.min()}, {img_bgr.max()}]")
print(f"Gray dimensions  : {img_gray.shape}")
print(f"Gray data type   : {img_gray.dtype}")
print(f"Gray intensity   : [{img_gray.min()}, {img_gray.max()}]")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(img_rgb)
axes[0].set_title("Original MRI (RGB)")
axes[0].axis("off")

axes[1].imshow(img_gray, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Grayscale (Luminance Formula)")
axes[1].axis("off")

plt.tight_layout()
plt.show()

##  Spatial Rescaling — Shrinking and Zooming


In [ ]:
# 50% shrinking using nearest-neighbor interpolation
half_w, half_h = w // 2, h // 2
shrunk_nn = cv2.resize(
    img_gray,
    (half_w, half_h),
    interpolation=cv2.INTER_NEAREST
)

# 2× enlargement
zoom_nn = cv2.resize(
    img_gray,
    (w * 2, h * 2),
    interpolation=cv2.INTER_NEAREST
)

zoom_bilinear = cv2.resize(
    img_gray,
    (w * 2, h * 2),
    interpolation=cv2.INTER_LINEAR
)

print(f"Original size       : {img_gray.shape}")
print(f"50% shrunk size     : {shrunk_nn.shape}")
print(f"2x nearest size     : {zoom_nn.shape}")
print(f"2x bilinear size    : {zoom_bilinear.shape}")

In [ ]:
# Multi-panel visualization: original vs. rescaled images
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

axes[0].imshow(img_gray, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")

axes[1].imshow(shrunk_nn, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("50% Shrink — Nearest")

axes[2].imshow(zoom_nn, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("2× Zoom — Nearest")

axes[3].imshow(zoom_bilinear, cmap="gray", vmin=0, vmax=255)
axes[3].set_title("2× Zoom — Bilinear")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

## Task 1 — Synthetic Noise Modeling



In [ ]:
sigma = 25

gaussian_noise = np.random.normal(
    loc=0.0,
    scale=sigma,
    size=img_gray.shape
).astype(np.float32)

noisy_gaussian = np.clip(
    img_gray.astype(np.float32) + gaussian_noise,
    0,
    255
).astype(np.uint8)

noisy_sp = img_gray.copy()

total_pixels = img_gray.size
num_corrupted = int(0.05 * total_pixels)

noise_indices = np.random.choice(
    total_pixels,
    size=num_corrupted,
    replace=False
)

salt_mask = np.random.rand(num_corrupted) < 0.5
flat_sp = noisy_sp.reshape(-1)

flat_sp[noise_indices[salt_mask]] = 255
flat_sp[noise_indices[~salt_mask]] = 0

noisy_sp = flat_sp.reshape(img_gray.shape)

print(f"Gaussian sigma              : {sigma}")
print(f"Salt-and-Pepper corrupted   : {num_corrupted:,} pixels")
print(f"Corruption percentage       : {100 * num_corrupted / total_pixels:.2f}%")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(img_gray, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original MRI")

axes[1].imshow(noisy_gaussian, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Gaussian Noise (σ=25)")

axes[2].imshow(noisy_sp, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("Salt-and-Pepper Noise (5%)")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

## Task 2 — Spatial Denoising and PSNR



In [ ]:
gaussian_box_filtered = cv2.blur(noisy_gaussian, (5, 5))
sp_median_filtered = cv2.medianBlur(noisy_sp, 5)

def calculate_psnr(original, processed):
    original = original.astype(np.float64)
    processed = processed.astype(np.float64)

    mse = np.mean((original - processed) ** 2)

    if mse == 0:
        return float("inf")

    return 10 * np.log10((255.0 ** 2) / mse)

psnr_gaussian_noisy = calculate_psnr(img_gray, noisy_gaussian)
psnr_gaussian_filtered = calculate_psnr(img_gray, gaussian_box_filtered)

psnr_sp_noisy = calculate_psnr(img_gray, noisy_sp)
psnr_sp_filtered = calculate_psnr(img_gray, sp_median_filtered)

print("=" * 60)
print("PSNR RESULTS")
print("=" * 60)
print(f"Gaussian noisy       PSNR: {psnr_gaussian_noisy:.2f} dB")
print(f"Gaussian + Box 5x5   PSNR: {psnr_gaussian_filtered:.2f} dB")
print(f"S&P noisy            PSNR: {psnr_sp_noisy:.2f} dB")
print(f"S&P + Median 5x5     PSNR: {psnr_sp_filtered:.2f} dB")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

axes[0].imshow(noisy_gaussian, cmap="gray", vmin=0, vmax=255)
axes[0].set_title(f"Gaussian Noisy\nPSNR={psnr_gaussian_noisy:.2f} dB")

axes[1].imshow(gaussian_box_filtered, cmap="gray", vmin=0, vmax=255)
axes[1].set_title(f"Box Averaging 5×5\nPSNR={psnr_gaussian_filtered:.2f} dB")

axes[2].imshow(noisy_sp, cmap="gray", vmin=0, vmax=255)
axes[2].set_title(f"S&P Noisy\nPSNR={psnr_sp_noisy:.2f} dB")

axes[3].imshow(sp_median_filtered, cmap="gray", vmin=0, vmax=255)
axes[3].set_title(f"Median 5×5\nPSNR={psnr_sp_filtered:.2f} dB")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

## Brightness Histogram and Global Histogram Equalization

A 1D brightness histogram counts how frequently each intensity level `g ∈ [0,255]` occurs. The task specifically requests `cv2.calcHist()`.

Global histogram equalization uses the image's intensity distribution/CDF to redistribute intensities over a wider dynamic range, improving global contrast. fileciteturn1file0L53-L58

In [ ]:
hist_original = cv2.calcHist(
    [img_gray],
    [0],
    None,
    [256],
    [0, 256]
)

equalized_img = cv2.equalizeHist(img_gray)

hist_equalized = cv2.calcHist(
    [equalized_img],
    [0],
    None,
    [256],
    [0, 256]
)

print(f"Original intensity range  : [{img_gray.min()}, {img_gray.max()}]")
print(f"Equalized intensity range : [{equalized_img.min()}, {equalized_img.max()}]")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

axes[0, 0].imshow(img_gray, cmap="gray", vmin=0, vmax=255)
axes[0, 0].set_title("Original Grayscale MRI")
axes[0, 0].axis("off")

axes[0, 1].imshow(equalized_img, cmap="gray", vmin=0, vmax=255)
axes[0, 1].set_title("Global Histogram Equalization")
axes[0, 1].axis("off")

axes[1, 0].plot(hist_original)
axes[1, 0].set_title("Original Brightness Histogram")
axes[1, 0].set_xlabel("Intensity")
axes[1, 0].set_ylabel("Frequency")
axes[1, 0].set_xlim([0, 255])

axes[1, 1].plot(hist_equalized)
axes[1, 1].set_title("Equalized Brightness Histogram")
axes[1, 1].set_xlabel("Intensity")
axes[1, 1].set_ylabel("Frequency")
axes[1, 1].set_xlim([0, 255])

plt.tight_layout()
plt.show()

## Feature Extraction — Sobel Gradient Magnitude vs. Canny



In [ ]:
sobel_x = cv2.Sobel(
    equalized_img,
    cv2.CV_64F,
    1, 0,
    ksize=3
)

sobel_y = cv2.Sobel(
    equalized_img,
    cv2.CV_64F,
    0, 1,
    ksize=3
)

sobel_magnitude = cv2.magnitude(sobel_x, sobel_y)

sobel_magnitude_u8 = cv2.normalize(
    sobel_magnitude,
    None,
    0,
    255,
    cv2.NORM_MINMAX
).astype(np.uint8)

canny_edges = cv2.Canny(
    equalized_img,
    threshold1=50,
    threshold2=150
)

print(f"Sobel magnitude range: [{sobel_magnitude.min():.2f}, {sobel_magnitude.max():.2f}]")
print(f"Canny edge pixels    : {np.count_nonzero(canny_edges):,}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(equalized_img, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Equalized MRI")

axes[1].imshow(sobel_magnitude_u8, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Sobel Gradient Magnitude")

axes[2].imshow(canny_edges, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("Canny Edge Detection")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

## Segmentation — Otsu Thresholding vs. K-Means



In [ ]:
otsu_threshold, otsu_mask = cv2.threshold(
    equalized_img,
    0,
    255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU
)

pixel_values = equalized_img.reshape(-1, 1).astype(np.float32)

k = 3
kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

labels = kmeans.fit_predict(pixel_values)
centers = kmeans.cluster_centers_.flatten()

sort_order = np.argsort(centers)
sorted_centers = centers[sort_order]

label_mapping = np.empty_like(sort_order)
label_mapping[sort_order] = np.arange(k)

sorted_labels = label_mapping[labels]
kmeans_map = sorted_labels.reshape(equalized_img.shape)

kmeans_intensity_map = sorted_centers[kmeans_map].astype(np.uint8)

print("=" * 60)
print("SEGMENTATION RESULTS")
print("=" * 60)
print(f"Otsu threshold T* : {otsu_threshold:.0f}")
print(f"K-Means K          : {k}")
print("K-Means centers    :", np.round(sorted_centers, 2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(equalized_img, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Equalized MRI")

axes[1].imshow(otsu_mask, cmap="gray", vmin=0, vmax=255)
axes[1].set_title(f"Otsu Binary Mask (T*={otsu_threshold:.0f})")

axes[2].imshow(kmeans_intensity_map, cmap="gray", vmin=0, vmax=255)
axes[2].set_title(f"K-Means Cluster Map (K={k})")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

##  Final Required Summary


In [ ]:
print("\n" + "=" * 72)
print("TASK 25 — FINAL PIPELINE SUMMARY")
print("=" * 72)

print(f"Dataset image       : {Path(image_path).name}")
print(f"Image dimensions    : H={h}, W={w}, C={c}")
print(f"Image dtype         : {img_bgr.dtype}")
print(f"Intensity range     : [{img_bgr.min()}, {img_bgr.max()}]")

print("\n--- Rescaling ---")
print(f"50% shrink          : {shrunk_nn.shape}")
print(f"2x nearest          : {zoom_nn.shape}")
print(f"2x bilinear         : {zoom_bilinear.shape}")

print("\n--- Noise / PSNR ---")
print(f"Gaussian noise σ=25 : {psnr_gaussian_noisy:.2f} dB")
print(f"Gaussian + Box 5x5  : {psnr_gaussian_filtered:.2f} dB")
print(f"S&P noise (5%)      : {psnr_sp_noisy:.2f} dB")
print(f"S&P + Median 5x5    : {psnr_sp_filtered:.2f} dB")

print("\n--- Contrast ---")
print(f"Original range      : [{img_gray.min()}, {img_gray.max()}]")
print(f"Equalized range     : [{equalized_img.min()}, {equalized_img.max()}]")

print("\n--- Segmentation ---")
print(f"Otsu threshold T*   : {otsu_threshold:.0f}")
print(f"K-Means K           : {k}")
print(f"K-Means centers     : {np.round(sorted_centers, 2)}")

print("=" * 72)

## 10. Conclusions

### Image representation
The MRI is loaded in OpenCV's BGR representation, converted to RGB for display, and transformed to an 8-bit grayscale image using the specified luminance equation.

### Rescaling
Nearest-neighbor shrinking performs direct subsampling. During 2× enlargement, nearest-neighbor creates visibly blockier structural boundaries, while bilinear interpolation produces smoother transitions.

### Noise and denoising
Gaussian noise is modeled with `σ=25`, while Salt-and-Pepper noise corrupts 5% of pixels. The 5×5 box filter is used for Gaussian noise and the 5×5 median filter for impulse noise, with PSNR providing a numerical fidelity measure.

### Contrast enhancement
Global histogram equalization redistributes grayscale intensities and can reveal structures that are difficult to distinguish in a low-contrast image.

### Feature extraction
Sobel gradient magnitude highlights regions of strong intensity change. Canny converts these changes into a more explicit binary edge representation.

### Segmentation
Otsu provides an automatically selected binary threshold, while K-Means produces multiple intensity-based regions. The two methods therefore provide complementary ways of isolating image regions.
